# Task XII: Quantum State Preparation via PPO Reinforcement Learning

**ML4SCI GSoC 2026 Evaluation Test**

## Overview
This notebook extends Task XI by replacing gradient-based MLP training with
**Proximal Policy Optimisation (PPO)** — a temporal difference RL algorithm.

The RL agent learns to adjust PQC parameters step-by-step to maximise fidelity
with a target quantum state, using negative MSE as the reward signal.

## RL Formulation
- **State** s_t : [current params θ_t (flattened), target descriptor (Re+Im statevector)]
- **Action** a_t : parameter adjustments Δθ ∈ [-Δ_max, Δ_max] per rotation angle
- **Reward** r_t : -MSE(|ψ(θ_t)|, |ψ_target|) — negative MSE = higher reward for better fidelity
- **Episode**: T=20 steps from a random initial θ, terminated early if fidelity > 0.99
- **Goal**: maximise cumulative reward = minimise MSE over the episode

## Why PPO over DQN
PQC parameter adjustments are continuous — PPO handles continuous action spaces
naturally via a Gaussian policy. DQN would require discretising each angle into
bins, losing precision. PPO's clipped surrogate objective also provides stable
training without the replay buffer instability of DQN.

## Architecture (Task XI MLP reused)
- **Actor**: 2 linear layers → outputs mean of Gaussian policy over Δθ
- **Critic**: 2 linear layers → estimates state value V(s)
- Both networks share the same input dimension as Task XI's MLP

## 0. Installation

In [ ]:
!pip install -q pennylane

## 1. Imports & Seeds

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Normal
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import random, os, pickle
import pennylane as qml

print(f'PennyLane : {qml.__version__}')
print(f'PyTorch   : {torch.__version__}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device('cpu')  # quantum circuit runs on CPU
print(f'Device: {DEVICE}')

## 2. Hyperparameters

In [ ]:
# ── Circuit ────────────────────────────────────────────────────────
N_QUBITS    = 4
N_LAYERS    = 3
N_PARAMS    = N_LAYERS * N_QUBITS * 2
STATE_DIM   = 2 ** N_QUBITS

# ── RL environment ─────────────────────────────────────────────────
T_MAX       = 20      # max steps per episode
DELTA_MAX   = 0.3     # max parameter adjustment per step (radians)
FIDELITY_THRESHOLD = 0.99  # early termination threshold

# ── PPO ────────────────────────────────────────────────────────────
N_EPISODES      = 300    # training episodes
PPO_EPOCHS      = 4      # PPO update epochs per rollout
CLIP_EPS        = 0.2    # PPO clipping epsilon
GAMMA           = 0.99   # discount factor
LAM             = 0.95   # GAE lambda
ACTOR_LR        = 3e-4
CRITIC_LR       = 1e-3
ENTROPY_COEF    = 0.01   # entropy bonus for exploration
VALUE_COEF      = 0.5    # value loss coefficient
MAX_GRAD_NORM   = 0.5
ROLLOUT_STEPS   = 5      # episodes per rollout before PPO update

# ── Evaluation ─────────────────────────────────────────────────────
N_EVAL_EPISODES = 20

# State dimension for actor/critic:
# current params (N_PARAMS) + target descriptor (2 * STATE_DIM)
OBS_DIM = N_PARAMS + 2 * STATE_DIM

print(f'N_PARAMS  : {N_PARAMS}')
print(f'STATE_DIM : {STATE_DIM}')
print(f'OBS_DIM   : {OBS_DIM}  (params + target descriptor)')
print(f'Action dim: {N_PARAMS}  (Δθ per angle)')

## 3. Quantum Circuit

Same PQC as Task XI — 4 qubits, 3 layers of RY+RZ+CNOT ring.
Returns statevector for MSE reward computation.
Uses `diff_method='backprop'` for compatibility with `qml.state()`.

In [ ]:
dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev, interface='torch', diff_method='backprop')
def pqc(params):
    """
    params: (N_LAYERS, N_QUBITS, 2) float64
    returns: statevector (STATE_DIM,) complex
    """
    for layer in range(N_LAYERS):
        for i in range(N_QUBITS):
            qml.RY(params[layer, i, 0], wires=i)
            qml.RZ(params[layer, i, 1], wires=i)
        for i in range(N_QUBITS):
            qml.CNOT(wires=[i, (i+1) % N_QUBITS])
    return qml.state()


def compute_fidelity(params, target_params):
    """Fidelity |<ψ(params)|ψ(target_params)>|^2 — no grad needed."""
    with torch.no_grad():
        psi_p = pqc(params)
        psi_t = pqc(target_params)
        return (psi_p.conj() * psi_t).sum().abs().item() ** 2


def compute_mse(params, target_params):
    """MSE between statevectors (real+imag) — no grad needed."""
    with torch.no_grad():
        psi_p = pqc(params)
        psi_t = pqc(target_params)
        pred  = torch.cat([psi_p.real, psi_p.imag])
        tgt   = torch.cat([psi_t.real, psi_t.imag])
        return F.mse_loss(pred, tgt).item()


def random_params():
    """Random PQC parameters in [0, 2π]."""
    return torch.tensor(
        np.random.uniform(0, 2*np.pi, (N_LAYERS, N_QUBITS, 2)),
        dtype=torch.float64
    )


def params_to_obs(params, target_params):
    """
    Build observation vector for actor/critic.
    Concatenates: [flattened current params, Re(ψ_target), Im(ψ_target)]
    Returns float32 tensor of shape (OBS_DIM,)
    """
    with torch.no_grad():
        sv_t = pqc(target_params)
    desc = torch.cat([
        params.float().view(-1),           # current params
        sv_t.real.float(),                 # target Re
        sv_t.imag.float()                  # target Im
    ])
    return desc


# Sanity check
tp = random_params()
p  = random_params()
print(f'Fidelity (random vs random): {compute_fidelity(p, tp):.4f}')
print(f'Fidelity (same params)     : {compute_fidelity(tp, tp):.4f}  (should be 1.0)')
print(f'Obs vector shape           : {params_to_obs(p, tp).shape}')

## 4. RL Environment

A lightweight Gym-style environment for quantum state preparation.

- **reset()**: sample new random initial params θ_0, keep fixed target θ_target
- **step(action)**: apply Δθ to current params, compute reward = -MSE
- **Reward shaping**: we add a fidelity bonus for reaching high fidelity,
  encouraging the agent to push all the way to fidelity=1 rather than
  stopping at a local minimum.

In [ ]:
class QuantumStateEnv:
    """
    RL environment for quantum state preparation.

    The agent controls PQC parameters θ by applying incremental adjustments.
    Reward = -MSE(|ψ(θ)|, |ψ_target|) at each step.
    Episode terminates after T_MAX steps or when fidelity > FIDELITY_THRESHOLD.
    """
    def __init__(self, target_params):
        self.target_params = target_params
        self.params        = None
        self.t             = 0

    def reset(self):
        """Reset to new random initial parameters."""
        self.params = random_params()
        self.t      = 0
        return params_to_obs(self.params, self.target_params)

    def step(self, action):
        """
        action: (N_PARAMS,) float32 tensor — parameter adjustments
        Returns: (next_obs, reward, done)
        """
        # Clip and apply action
        delta       = action.double().view(N_LAYERS, N_QUBITS, 2).clamp(-DELTA_MAX, DELTA_MAX)
        self.params = (self.params + delta) % (2 * np.pi)
        self.t     += 1

        # Compute reward = -MSE (higher = better)
        mse     = compute_mse(self.params, self.target_params)
        fid     = compute_fidelity(self.params, self.target_params)

        # Reward = -MSE + fidelity bonus for high fidelity
        reward  = -mse + 0.1 * fid

        done    = (self.t >= T_MAX) or (fid >= FIDELITY_THRESHOLD)
        next_obs = params_to_obs(self.params, self.target_params)

        return next_obs, reward, done, {'fidelity': fid, 'mse': mse}

    def get_fidelity(self):
        return compute_fidelity(self.params, self.target_params)


# Test environment
tp_test  = random_params()
env_test = QuantumStateEnv(tp_test)
obs      = env_test.reset()
print(f'Initial obs shape : {obs.shape}')
dummy_act = torch.zeros(N_PARAMS)
obs2, r, done, info = env_test.step(dummy_act)
print(f'Reward            : {r:.4f}')
print(f'Initial fidelity  : {info["fidelity"]:.4f}')
print(f'Done              : {done}')

## 5. PPO Actor-Critic Networks

Both actor and critic use the same 2-hidden-layer MLP architecture as Task XI.

**Actor**: outputs mean μ of a Gaussian policy N(μ, σ²) over parameter adjustments Δθ.
σ is a learned log-std parameter (not input-dependent, standard PPO practice).
Actions are sampled from this distribution during rollout and used with the
log-probability for the PPO surrogate loss.

**Critic**: outputs scalar state value V(s) used for advantage estimation.

**PPO objective**:
L_CLIP = E[min(r_t A_t, clip(r_t, 1-ε, 1+ε) A_t)]
where r_t = π(a|s) / π_old(a|s) is the probability ratio and A_t is the GAE advantage.

In [ ]:
class Actor(nn.Module):
    """
    PPO Actor — Gaussian policy over PQC parameter adjustments.
    Architecture: Linear(OBS_DIM->64)->ReLU->Linear(64->64)->ReLU->Linear(64->N_PARAMS)
    """
    def __init__(self, obs_dim=OBS_DIM, act_dim=N_PARAMS, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, act_dim), nn.Tanh()  # mean in (-1,1), scaled by DELTA_MAX
        )
        # Learned log std — initialised to log(0.5) for moderate exploration
        self.log_std = nn.Parameter(torch.full((act_dim,), np.log(0.5)))

    def forward(self, obs):
        mean = self.net(obs) * DELTA_MAX   # scale to [-DELTA_MAX, DELTA_MAX]
        std  = self.log_std.exp().clamp(1e-4, 1.0)
        return Normal(mean, std)

    def get_action(self, obs):
        dist   = self.forward(obs)
        action = dist.sample()
        log_p  = dist.log_prob(action).sum(-1)
        return action, log_p

    def evaluate_action(self, obs, action):
        dist   = self.forward(obs)
        log_p  = dist.log_prob(action).sum(-1)
        entropy= dist.entropy().sum(-1)
        return log_p, entropy


class Critic(nn.Module):
    """
    PPO Critic — estimates state value V(s).
    Same architecture as Actor but outputs scalar.
    """
    def __init__(self, obs_dim=OBS_DIM, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),  nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, obs):
        return self.net(obs).squeeze(-1)


actor  = Actor()
critic = Critic()
actor_opt  = optim.Adam(actor.parameters(),  lr=ACTOR_LR)
critic_opt = optim.Adam(critic.parameters(), lr=CRITIC_LR)

print(f'Actor params : {sum(p.numel() for p in actor.parameters()):,}')
print(f'Critic params: {sum(p.numel() for p in critic.parameters()):,}')

# Test forward pass
test_obs  = params_to_obs(random_params(), random_params())
dist_test = actor(test_obs.unsqueeze(0))
print(f'Policy mean shape: {dist_test.mean.shape}  (1, N_PARAMS)')
print(f'Value estimate   : {critic(test_obs.unsqueeze(0)).item():.4f}')

## 6. PPO Training

### Rollout collection
For each rollout, the agent interacts with the environment for ROLLOUT_STEPS episodes,
collecting (obs, action, log_prob, reward, value, done) tuples.

### GAE Advantage estimation
Generalised Advantage Estimation (GAE) reduces variance while controlling bias:
A_t = Σ (γλ)^k δ_{t+k}  where δ_t = r_t + γV(s_{t+1}) - V(s_t)

### PPO update
For PPO_EPOCHS epochs over the collected rollout:
- Compute probability ratios r_t = π_new / π_old
- Clipped surrogate loss: L_CLIP = min(r_t A_t, clip(r_t, 1-ε, 1+ε) A_t)
- Value loss: L_V = (V(s) - returns)²
- Entropy bonus: -H(π) encourages exploration
- Total: L = -L_CLIP + VALUE_COEF * L_V - ENTROPY_COEF * H

In [ ]:
def compute_gae(rewards, values, dones, last_value, gamma=GAMMA, lam=LAM):
    """
    Compute GAE advantages and returns.
    rewards, values, dones: lists of scalars
    last_value: V(s_T) for bootstrapping
    """
    advantages = []
    gae        = 0.0
    next_val   = last_value

    for t in reversed(range(len(rewards))):
        mask   = 1.0 - dones[t]
        delta  = rewards[t] + gamma * next_val * mask - values[t]
        gae    = delta + gamma * lam * mask * gae
        advantages.insert(0, gae)
        next_val = values[t]

    advantages = torch.tensor(advantages, dtype=torch.float32)
    returns    = advantages + torch.tensor(values, dtype=torch.float32)
    # Normalise advantages
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
    return advantages, returns


def ppo_update(obs_buf, act_buf, logp_buf, adv_buf, ret_buf):
    """
    Run PPO_EPOCHS of PPO updates on the collected rollout.
    Returns (actor_loss, critic_loss, entropy) for logging.
    """
    obs_t  = torch.stack(obs_buf)
    act_t  = torch.stack(act_buf)
    logp_t = torch.stack(logp_buf).detach()

    total_al = total_cl = total_ent = 0.0

    for _ in range(PPO_EPOCHS):
        # Actor update
        new_logp, entropy = actor.evaluate_action(obs_t, act_t)
        ratio  = (new_logp - logp_t).exp()
        clip_r = ratio.clamp(1 - CLIP_EPS, 1 + CLIP_EPS)
        actor_loss = -torch.min(ratio * adv_buf, clip_r * adv_buf).mean()
        entropy_loss = -entropy.mean()
        actor_total  = actor_loss + ENTROPY_COEF * entropy_loss

        actor_opt.zero_grad()
        actor_total.backward()
        nn.utils.clip_grad_norm_(actor.parameters(), MAX_GRAD_NORM)
        actor_opt.step()

        # Critic update
        values_pred  = critic(obs_t)
        critic_loss  = VALUE_COEF * F.mse_loss(values_pred, ret_buf)

        critic_opt.zero_grad()
        critic_loss.backward()
        nn.utils.clip_grad_norm_(critic.parameters(), MAX_GRAD_NORM)
        critic_opt.step()

        total_al  += actor_loss.item()
        total_cl  += critic_loss.item()
        total_ent += entropy.mean().item()

    n = PPO_EPOCHS
    return total_al/n, total_cl/n, total_ent/n

In [ ]:
# Fixed target state for training
target_params = random_params()
env           = QuantumStateEnv(target_params)

history = {
    'episode_reward': [], 'episode_fidelity': [],
    'actor_loss': [],     'critic_loss': [],
    'entropy': []
}

print(f'PPO training — {N_EPISODES} episodes...')
print(f'Target state fidelity (random vs target): {compute_fidelity(random_params(), target_params):.4f}')

# Rollout buffers
obs_buf  = []; act_buf  = []; logp_buf = []
rew_buf  = []; val_buf  = []; done_buf = []

for ep in tqdm(range(1, N_EPISODES+1)):
    obs      = env.reset()
    ep_rew   = 0.0
    ep_fid   = 0.0

    for t in range(T_MAX):
        obs_t   = obs.unsqueeze(0)

        with torch.no_grad():
            action, log_p = actor.get_action(obs_t)
            value         = critic(obs_t).item()

        next_obs, reward, done, info = env.step(action.squeeze(0))

        obs_buf.append(obs)
        act_buf.append(action.squeeze(0))
        logp_buf.append(log_p.squeeze(0))
        rew_buf.append(reward)
        val_buf.append(value)
        done_buf.append(float(done))

        obs      = next_obs
        ep_rew  += reward
        ep_fid   = info['fidelity']

        if done: break

    history['episode_reward'].append(ep_rew)
    history['episode_fidelity'].append(ep_fid)

    # PPO update every ROLLOUT_STEPS episodes
    if ep % ROLLOUT_STEPS == 0 and len(obs_buf) > 0:
        with torch.no_grad():
            last_val = critic(obs.unsqueeze(0)).item()
        adv, ret = compute_gae(rew_buf, val_buf, done_buf, last_val)
        al, cl, ent = ppo_update(obs_buf, act_buf, logp_buf, adv, ret)
        history['actor_loss'].append(al)
        history['critic_loss'].append(cl)
        history['entropy'].append(ent)
        # Clear buffers
        obs_buf=[]; act_buf=[]; logp_buf=[]
        rew_buf=[]; val_buf=[]; done_buf=[]

    if ep % 50 == 0:
        recent_fid = np.mean(history['episode_fidelity'][-20:])
        recent_rew = np.mean(history['episode_reward'][-20:])
        print(f'Ep {ep:4d} | mean_fid {recent_fid:.4f} | mean_rew {recent_rew:.4f}')

print('\nTraining complete.')
print(f'Final mean fidelity (last 20 eps): {np.mean(history["episode_fidelity"][-20:]):.4f}')

## 7. Evaluation — PPO vs Gradient Descent (Task XI)

We compare PPO against gradient-based optimisation (Task XI baseline)
on new held-out target states. Both methods get the same number of
function evaluations (circuit calls) for a fair comparison.

In [ ]:
def gradient_baseline(target_params, n_steps=T_MAX, lr=0.05):
    """
    Direct gradient descent on PQC parameters (Task XI approach).
    Returns fidelity curve over steps.
    """
    params = random_params().requires_grad_(True)
    opt    = optim.Adam([params], lr=lr)
    fids   = []
    for _ in range(n_steps):
        opt.zero_grad()
        psi_p  = pqc(params)
        psi_t  = pqc(target_params)
        pred   = torch.cat([psi_p.real, psi_p.imag])
        tgt    = torch.cat([psi_t.real, psi_t.imag]).detach()
        loss   = F.mse_loss(pred, tgt)
        loss.backward()
        opt.step()
        with torch.no_grad():
            fids.append(compute_fidelity(params.detach(), target_params))
    return fids


def ppo_eval(target_params, n_steps=T_MAX):
    """
    Run trained PPO agent on a new target state.
    Returns fidelity curve over steps.
    """
    eval_env = QuantumStateEnv(target_params)
    obs      = eval_env.reset()
    fids     = []
    for _ in range(n_steps):
        with torch.no_grad():
            action, _ = actor.get_action(obs.unsqueeze(0))
        obs, _, done, info = eval_env.step(action.squeeze(0))
        fids.append(info['fidelity'])
        if done: break
    # Pad to n_steps if terminated early
    while len(fids) < n_steps:
        fids.append(fids[-1])
    return fids


print(f'Evaluating on {N_EVAL_EPISODES} held-out target states...')
ppo_fids  = []
grad_fids = []

for i in tqdm(range(N_EVAL_EPISODES)):
    tp = random_params()
    ppo_fids.append(ppo_eval(tp))
    grad_fids.append(gradient_baseline(tp))

ppo_curve  = np.mean(ppo_fids,  axis=0)
grad_curve = np.mean(grad_fids, axis=0)

print(f'\nFinal fidelity (avg over {N_EVAL_EPISODES} held-out states):')
print(f'  Gradient descent (Task XI): {grad_curve[-1]:.4f}')
print(f'  PPO (Task XII)            : {ppo_curve[-1]:.4f}')

## 8. Plots & Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. PPO training curves ─────────────────────────────────────────
# Smooth fidelity with rolling mean
def smooth(x, w=20):
    return np.convolve(x, np.ones(w)/w, mode='valid')

fid_sm = smooth(history['episode_fidelity'])
rew_sm = smooth(history['episode_reward'])
ep_ax  = range(len(fid_sm))

ax  = axes[0]
ax2 = ax.twinx()
ax.plot(ep_ax,  fid_sm, color='steelblue',  lw=2, label='Fidelity (smoothed)')
ax2.plot(ep_ax, rew_sm, color='darkorange', lw=1.5, alpha=0.7, linestyle='--', label='Reward (smoothed)')
ax.set_xlabel('Episode'); ax.set_ylabel('Fidelity', color='steelblue')
ax2.set_ylabel('Episode reward', color='darkorange')
ax.set_title('PPO training')
ax.set_ylim(0, 1)
l1,lb1=ax.get_legend_handles_labels(); l2,lb2=ax2.get_legend_handles_labels()
ax.legend(l1+l2, lb1+lb2, fontsize=9)
ax.grid(True, alpha=0.3)

# ── 2. PPO loss curves ─────────────────────────────────────────────
upd = range(len(history['actor_loss']))
axes[1].plot(upd, history['actor_loss'],  color='steelblue',  lw=2, label='Actor loss')
axes[1].plot(upd, history['critic_loss'], color='darkorange', lw=2, label='Critic loss')
axes[1].set_xlabel('PPO update'); axes[1].set_ylabel('Loss')
axes[1].set_title('PPO actor & critic losses')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

# ── 3. PPO vs gradient descent fidelity curves ─────────────────────
steps = range(1, len(ppo_curve)+1)
axes[2].plot(steps, grad_curve, color='tomato',    lw=2, label=f'Gradient descent (Task XI) — final: {grad_curve[-1]:.4f}')
axes[2].plot(steps, ppo_curve,  color='steelblue', lw=2, label=f'PPO (Task XII) — final: {ppo_curve[-1]:.4f}')
axes[2].fill_between(steps,
    np.mean(ppo_fids, axis=0) - np.std(ppo_fids, axis=0),
    np.mean(ppo_fids, axis=0) + np.std(ppo_fids, axis=0),
    alpha=0.2, color='steelblue')
axes[2].fill_between(steps,
    np.mean(grad_fids, axis=0) - np.std(grad_fids, axis=0),
    np.mean(grad_fids, axis=0) + np.std(grad_fids, axis=0),
    alpha=0.2, color='tomato')
axes[2].set_xlabel('Steps'); axes[2].set_ylabel('Fidelity |<ψ̂|ψ_target>|²')
axes[2].set_title(f'PPO vs gradient descent\n({N_EVAL_EPISODES} held-out target states)')
axes[2].set_ylim(0, 1); axes[2].legend(fontsize=9); axes[2].grid(True, alpha=0.3)

plt.suptitle('Task XII: Quantum State Preparation via PPO', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('task_xii_results.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Results

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE_DIR = '/content/drive/MyDrive/ml4sci_data'

torch.save(actor.state_dict(),  f'{DRIVE_DIR}/task_xii_actor.pt')
torch.save(critic.state_dict(), f'{DRIVE_DIR}/task_xii_critic.pt')
with open(f'{DRIVE_DIR}/task_xii_history.pkl', 'wb') as f:
    pickle.dump({'history': history, 'ppo_fids': ppo_fids, 'grad_fids': grad_fids}, f)
fig.savefig(f'{DRIVE_DIR}/task_xii_results.png', dpi=150, bbox_inches='tight')
print('Saved to Drive.')

## 10. Discussion

### Why RL for quantum state preparation?
Gradient-based optimisation (Task XI) computes parameter gradients directly through
the quantum circuit. This works well when the loss landscape is smooth but fails
in barren plateau regions where gradients vanish exponentially with circuit depth.
PPO explores the parameter space through stochastic action sampling rather than
following gradients, making it potentially more robust to flat loss landscapes.

### PPO formulation for quantum control
The key design choice is treating parameter *adjustments* Δθ (not absolute values)
as actions. This makes the action space stationary — the agent learns a policy for
how to move through parameter space regardless of where it currently is. The
Gaussian policy with learned log-std balances exploration (high std early) with
exploitation (low std once near the target).

### MSE in the reward function
Reward = -MSE(|ψ(θ)|, |ψ_target|) + 0.1 × fidelity. The MSE term provides
dense per-step feedback. The fidelity bonus encourages the agent to push toward
high-fidelity solutions rather than settling for local minima with low MSE.

### Comparison with Task XI
Gradient descent typically converges faster per step when gradients are available
and informative. PPO's advantage is in settings where: (1) gradient computation
is expensive or unavailable (hardware QPUs), (2) the landscape has many local
minima, or (3) the agent needs to generalise across many target states without
retraining — which connects naturally to the Q-MAML framework from Task XI.